## AI Flight assistant with set_ticket_price tool

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

import sqlite3

DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"
def set_ticket_price(city, price):
    print(f"DATABASE TOOL CALLED: Setting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()
    return f"Ticket price to {city} has been set to ${price}"
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999,"jakarta":699,"moscow":1399}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)
get_ticket_price("moscow")

### Wrapper functions for the tools

In [ ]:
def get_ticket_price_tool(destination_city):
    return get_ticket_price(destination_city)


def set_ticket_price_tool(destination_city, price):
    return set_ticket_price(destination_city, price)
available_functions = {
    "get_ticket_price": get_ticket_price_tool,
    "set_ticket_price": set_ticket_price_tool,
}

In [ ]:
load_dotenv(override=True)
MODEL = "gemma4:e4b"
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.

You can look up ticket prices using the get_ticket_price tool.
You can update ticket prices using the set_ticket_price tool only when the user clearly provides both a destination city and a price.
"""

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

set_price_function = {
    "name": "set_ticket_price",
    "description": "Set or update the price of a return ticket to a destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The destination city whose ticket price should be updated",
            },
            "price": {
                "type": "number",
                "description": "The new ticket price for the destination city",
            },
        },
        "required": ["destination_city", "price"],
        "additionalProperties": False
    }
}

tools = [
    {"type": "function", "function": price_function},
    {"type": "function", "function": set_price_function}
]

##handle tool calls can handle any number of tool calls, the code is now more "pythony"

def handle_tool_calls(message):
    responses = []

    print(message.tool_calls)

    for tool_call in message.tool_calls:
        function_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        function_to_call = available_functions.get(function_name)

        if function_to_call:
            result = function_to_call(**arguments)
        else:
            result = f"Unknown tool: {function_name}"

        responses.append({
            "role": "tool",
            "content": str(result),
            "tool_call_id": tool_call.id
        })

    return responses
    
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = ollama.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()